In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/RxHandBD-ML")

TRAIN_CSV = DATA_DIR / "Train_Label.csv"
TEST_CSV = DATA_DIR / "Test_Label.csv"

TRAIN_IMG_DIR = DATA_DIR / "Train_Set"
TEST_IMG_DIR = DATA_DIR / "Test_Set"

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.isnull().sum()

In [ ]:
train_df.duplicated().sum()

In [ ]:
print("\nTraining images:", len(list(TRAIN_IMG_DIR.glob("*.jpg"))))
print("Testing images:", len(list(TEST_IMG_DIR.glob("*.jpg"))))

Displaying some images with their actual text labels.

In [ ]:
plt.figure(figsize=(15, 8))

for i in range(9):
    row = train_df.iloc[i]

    image_path = TRAIN_IMG_DIR / row["Images"]
    image = Image.open(image_path).convert("L")

    plt.subplot(3, 3, i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(row["Text"])
    plt.axis("off")

plt.tight_layout()
plt.show()

**Divide training data into training and validation**

Why use drop=True?

It prevents pandas from adding the old index as a new column.

In simple words: We reset the index to make row numbering start from 0 and continue sequentially.

In [ ]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.10,
    random_state=42
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Testing samples:", len(test_df))

Convert text into numbers

In [ ]:
characters = sorted(set("".join(train_data["Text"].astype(str))))

char_to_num = {char: i + 1 for i, char in enumerate(characters)}
num_to_char = {i + 1: char for i, char in enumerate(characters)}

print("Total characters:", len(characters))
print(char_to_num)

Image preprocessing

In [ ]:
IMG_HEIGHT = 32
IMG_WIDTH = 256

def preprocess_image(image_path):

    image = Image.open(image_path).convert("L")

    width, height = image.size

    new_width = int(width * IMG_HEIGHT / height)
    new_width = min(new_width, IMG_WIDTH)
    new_width = max(new_width, 1)

    image = image.resize((new_width, IMG_HEIGHT))

    image = np.array(image, dtype=np.float32) / 255.0

    # Create white background
    padded_image = np.ones((IMG_HEIGHT, IMG_WIDTH))

    # Place resized handwriting on the left
    padded_image[:, :new_width] = image

    return padded_image

In [ ]:
sample_path = TRAIN_IMG_DIR / train_data.iloc[0]["Images"]

processed_image = preprocess_image(sample_path)

plt.figure(figsize=(10, 3))
plt.imshow(processed_image, cmap="gray")
plt.title("Preprocessed Image")
plt.axis("off")
plt.show()

In [ ]:
print("Processed image shape:", processed_image.shape)
print("Minimum pixel value:", processed_image.min())
print("Maximum pixel value:", processed_image.max())

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(9):
    sample_path = TRAIN_IMG_DIR / train_data.iloc[i]["Images"]
    processed_image = preprocess_image(sample_path)

    plt.subplot(3, 3, i + 1)
    plt.imshow(processed_image, cmap="gray")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
for i in range(9):
    print("Image:", train_data.iloc[i]["Images"])
    print("Actual Text:", train_data.iloc[i]["Text"])
    print("----------------------")

Convert text labels into sequences
What are we doing?

Converting each word into a sequence of numbers using the character mapping we created earlier.

In [ ]:
def encode_text(text):

    return [char_to_num[char] for char in text]

sample_text = train_data.iloc[0]["Text"]

print("Original text:", sample_text)
print("Encoded text:", encode_text(sample_text))

**Prepare the data for TensorFlow**
Combining the processed images and encoded text into batches so that TensorFlow can train on them.

In simple words: This step creates an organized pipeline that loads images, prepares their labels, groups examples into batches, and feeds them to our CRNN model.

It does not train the model yet. It only prepares the data for training.

In [ ]:
def create_generator(data, image_folder):

    for _, row in data.iterrows():

        image_path = image_folder / row["Images"]

        image = preprocess_image(image_path)

        label = encode_text(row["Text"])

        original_width = Image.open(image_path).width
        original_height = Image.open(image_path).height

        resized_width = min(
            max(1, int(original_width * IMG_HEIGHT / original_height)),
            IMG_WIDTH
        )

        input_length = int(np.ceil(np.ceil(resized_width / 2) / 2))

        yield (
            image[..., np.newaxis],
            np.array(label, dtype=np.int32),
            np.int32(input_length),
            row["Text"],
            str(image_path)
        )

Create TensorFlow datasets:

In [ ]:
max_label_length = max(
    train_data["Text"].str.len().max(),
    val_data["Text"].str.len().max(),
    test_df["Text"].str.len().max()
)

output_signature = (
    tf.TensorSpec((IMG_HEIGHT, IMG_WIDTH, 1), tf.float32),
    tf.TensorSpec((None,), tf.int32),
    tf.TensorSpec((), tf.int32),
    tf.TensorSpec((), tf.string),
    tf.TensorSpec((), tf.string)
)

def create_dataset(data, image_folder, training=False):

    dataset = tf.data.Dataset.from_generator(
        lambda: create_generator(data, image_folder),
        output_signature=output_signature
    )

    dataset = dataset.padded_batch(
        32,
        padded_shapes=(
            [IMG_HEIGHT, IMG_WIDTH, 1],
            [None],
            [],
            [],
            []
        ),
        padding_values=(
            0.0,
            -1,
            0,
            "",
            ""
        )
    )

    if training:
        dataset = dataset.shuffle(500)

    return dataset.prefetch(tf.data.AUTOTUNE)


train_ds = create_dataset(train_data, TRAIN_IMG_DIR, True)
val_ds = create_dataset(val_data, TRAIN_IMG_DIR)
test_ds = create_dataset(test_df, TEST_IMG_DIR)

print("Datasets are ready!")

**Build the CRNN model**
What are we doing?

Now we build the actual model.

Our CRNN has:
- CNN to identify image features.
- BiLSTM to understand character sequences.
- Dense layer to predict characters.

First, CNN:

In [ ]:
from tensorflow.keras import layers, Model

input_layer = layers.Input(
    shape=(IMG_HEIGHT, IMG_WIDTH, 1)
)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(input_layer)
x = layers.MaxPooling2D((2, 2), padding="same")(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D((2, 2), padding="same")(x)

x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D((2, 1), padding="same")(x)

x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D((2, 1), padding="same")(x)

x = layers.Conv2D(256, (2, 1), padding="valid", activation="relu")(x)

x = layers.Reshape((64, 256))(x)

Now add BiLSTM:

In [ ]:
x = layers.Bidirectional(
    layers.LSTM(128, return_sequences=True)
)(x)

x = layers.Bidirectional(
    layers.LSTM(128, return_sequences=True)
)(x)

Finally, add the output layer:


In [ ]:
num_classes = len(characters) + 1

output = layers.Dense(num_classes)(x)

model = Model(input_layer, output)

model.summary()

**Define CTC loss**

In [ ]:
class CRNNModel(tf.keras.Model):

    def __init__(self, model):
        super().__init__()
        self.model = model

    def call(self, images, training=False):
      return self.model(images, training=training)

    def train_step(self, data):

        images, labels, input_length, _, _ = data

        label_length = tf.reduce_sum(
            tf.cast(labels >= 0, tf.int32), axis=1
        )

        labels = tf.maximum(labels, 0)

        with tf.GradientTape() as tape:

            predictions = self(images, training=True)

            loss = tf.nn.ctc_loss(
                labels=labels,
                logits=predictions,
                label_length=label_length,
                logit_length=input_length,
                logits_time_major=False,
                blank_index=0
            )

            loss = tf.reduce_mean(loss)

        gradients = tape.gradient(loss, self.trainable_variables)

        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables)
        )

        return {"loss": loss}

    def test_step(self, data):

        images, labels, input_length, _, _ = data

        label_length = tf.reduce_sum(
            tf.cast(labels >= 0, tf.int32), axis=1
        )

        labels = tf.maximum(labels, 0)

        predictions = self(images, training=False)

        loss = tf.nn.ctc_loss(
            labels=labels,
            logits=predictions,
            label_length=label_length,
            logit_length=input_length,
            logits_time_major=False,
            blank_index=0
        )

        return {"loss": tf.reduce_mean(loss)}

**Compile and train the model**

In [ ]:
crnn = CRNNModel(model)

crnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
)

In [ ]:
history = crnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]
)